# 🚀 Crypto Alpha Engine: High-Speed Colab Ingestion to Google Drive

This notebook downloads historical market data from **Binance Data Vision (S3)** directly into **Google Drive** using Google's datacenter backbone (1 Gbps+).

* **Zero local disk space used.**
* **Zero home internet bandwidth consumed.**
* Saves partitioned Parquet files directly to `MyDrive/trading/crypto-alpha-engine/data/`.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE_DIR = '/content/drive/MyDrive/trading/crypto-alpha-engine/data'
os.makedirs(DRIVE_BASE_DIR, exist_ok=True)
print(f"Target Storage Directory: {DRIVE_BASE_DIR}")

In [ ]:
# 2. Clone Repository and Install Dependencies
!git clone https://github.com/sanjmen/crypto-alpha-engine.git /content/crypto-alpha-engine || (cd /content/crypto-alpha-engine && git pull)
%cd /content/crypto-alpha-engine
!pip install -r requirements.txt --quiet

In [ ]:
# 3. Execute Parallel Bulk Download Directly into Google Drive
import sys
sys.path.insert(0, '/content/crypto-alpha-engine')

from src.data.binance_vision_downloader import BinanceVisionDownloader
from src.data.ccxt_provider import TOP_30_SYMBOLS
from src.domain.enums import Timeframe

downloader = BinanceVisionDownloader(output_dir=DRIVE_BASE_DIR)

# Select universe and date range
SYMBOLS = TOP_30_SYMBOLS[:10]  # Top 10 crypto assets
YEARS = [2023, 2024, 2025, 2026]
TIMEFRAME = Timeframe.H1

print(f"Starting bulk download for {len(SYMBOLS)} assets across years {YEARS}...")

for symbol in SYMBOLS:
    print(f"\n=== Downloading {symbol} ({TIMEFRAME.value}) ===")
    for year in YEARS:
        for month in range(1, 13):
            df = downloader.download_monthly_klines(symbol=symbol, year=year, month=month, timeframe=TIMEFRAME)
            if df is not None and not df.empty:
                count = downloader.save_klines_to_parquet(symbol=symbol, timeframe=TIMEFRAME, df=df)
                print(f"  {year}-{month:02d}: +{len(df):,} bars (total {count:,} in Parquet)")

print("\nAll downloads completed and safely stored in Google Drive!")

In [ ]:
# 4. Fast Query Test using DuckDB on Google Drive Parquet Files
import duckdb

btc_parquet = f"{DRIVE_BASE_DIR}/bars/1h/BTC_USDT.parquet"
if os.path.exists(btc_parquet):
    print("Preview of BTC/USDT stored on Google Drive:")
    df_check = duckdb.query(f"SELECT to_timestamp(timestamp) as time, open, high, low, close, volume FROM read_parquet('{btc_parquet}') LIMIT 5").to_df()
    display(df_check)
else:
    print(f"File {btc_parquet} not found yet.")